In [1]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from datetime import datetime
import os
from bson.decimal128 import Decimal128
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import pandas as pd
load_dotenv(find_dotenv())


True

# Sales

In [2]:

engine = create_engine(os.environ['SQLSERVER_URI'])
client = MongoClient(os.environ['MONGO_URI'], server_api=ServerApi('1'))
cm_env = os.environ.get('CM_ENV', 'dev')

if cm_env == 'prod':
    collection = client["campomaq"]["cm_ventas"]
else:
    collection = client["campomaq_test"]["cm_ventas"]
# Create a new client and connect to the server

# Get the latest document (by FACTURA then FECHA)
last_doc = collection.find_one(sort=[("factura", -1), ("fecha", -1)])

# Fallback if no data
last_factura = last_doc["factura"] if last_doc else 0
last_fecha = last_doc["fecha"] if last_doc else datetime(2000, 1, 1)

print("Last FACTURA:", last_factura)
print("Last FECHA:", last_fecha)

# Read new data query

new_data_query = f"""SELECT * FROM cm_ventas WHERE FACTURA > {last_factura} OR (FACTURA = {last_factura} AND FECHA > '{last_fecha}')"""
df_update = pd.read_sql(new_data_query, engine)
df_update["FECHA"] = pd.to_datetime(df_update["FECHA"]) 

# Standardize column names: lowercase, replace spaces and special chars with underscores
def standardize_col_names(df):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r'[^0-9a-zA-Z]+', '_', regex=True)
    )
    return df

df_update = standardize_col_names(df_update)
print('Standardized columns:', df_update.columns.tolist())

# Convert FACTURA to int

df_update['factura'] = df_update['factura'].astype(int)


Last FACTURA: 55044
Last FECHA: 2025-09-10 00:00:00
Standardized columns: ['factura', 'fecha', 'cliente', 'total', 'valor_ncred', 'valor_ret', 'forma_pago']


In [3]:

if not df_update.empty:
    # Convert DataFrame to JSON
    records = df_update.to_dict(orient="records")
        
    collection.insert_many(records)
    print(f"✅ Inserted {len(records)} records into MongoDB")

else:
    print("No new data to insert.")

✅ Inserted 142 records into MongoDB


## Sales by product

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import pandas as pd
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# Load environment variables
load_dotenv(find_dotenv())

# SQL and MongoDB connections
engine = create_engine(os.environ['SQLSERVER_URI'])
client = MongoClient(os.environ['MONGO_URI'], server_api=ServerApi('1'), socketTimeoutMS=300000, connectTimeoutMS=300000)
cm_env = os.environ.get('CM_ENV', 'dev')

if cm_env == 'prod':
    collection = client["campomaq"]["cm_ventas_prods"]
else:
    collection = client["campomaq_test"]["cm_ventas_prods"]


# Get the latest document (by FACTURA then FECHA)
last_doc = collection.find_one(sort=[("invoice_number", -1), ("invoice_date", -1)])

# Fallback if no data
last_factura = last_doc["invoice_number"] if last_doc else 0
last_fecha = last_doc["invoice_date"] if last_doc else datetime(2000, 1, 1)

print("Last FACTURA:", last_factura)
print("Last FECHA:", last_fecha)


# Read all products from SQL Server
# df_product_sales = pd.read_sql("SELECT * FROM CM_VTAS_DETALLE WHERE FECHA>='2023-01-01'", engine)

# Read new data query

new_data_query = f"""SELECT * FROM CM_VTAS_DETALLE WHERE invoice_number > {last_factura} OR (invoice_number = {last_factura} AND invoice_date > '{last_fecha}')"""
df_product_sales = pd.read_sql(new_data_query, engine)
df_product_sales["invoice_date"] = pd.to_datetime(df_product_sales["invoice_date"]) 


# Convert FACTURA to int

df_product_sales['invoice_number'] = df_product_sales['invoice_number'].astype(int)
df_product_sales['invoice_item_id'] = df_product_sales['invoice_item_id'].astype(int)
df_product_sales['quantity'] = df_product_sales['quantity'].astype(int)
df_product_sales['product_name'] = df_product_sales['product_name'].str.strip()
df_product_sales['client_name'] = df_product_sales['client_name'].str.strip()
df_product_sales['brand_name'] = df_product_sales['brand_name'].str.strip()
df_product_sales['brand_name'] = df_product_sales['brand_name'].str.strip()
df_product_sales['category_name'] = df_product_sales['category_name'].str.strip()
df_product_sales['product_code'] = df_product_sales['product_code'].str.strip()


# BATCH INSERT FUNCTION
# def batch_insert(records, batch_size=1000):
#     for i in range(0, len(records), batch_size):
#         collection.insert_many(records[i:i+batch_size])

# # Usage
# if not df_product_sales.empty:
#     records = df_product_sales.to_dict(orient="records")
#     batch_insert(records)
#     print(f"✅ Inserted {len(records)} records into MongoDB in batches")
# else:
#     print("No new data to insert.")

# Check if there are new records to insert


if not df_product_sales.empty:
    # Convert DataFrame to JSON
    records = df_product_sales.to_dict(orient="records")
    collection.insert_many(records)
    print(f"✅ Inserted {len(records)} records into MongoDB")

else:
    print("No new data to insert.")


Last FACTURA: 55053
Last FECHA: 2025-09-10 00:00:00
✅ Inserted 445 records into MongoDB


In [5]:
df_product_sales

,invoice_item_id,invoice_number,invoice_date,product_code,product_name,quantity,sale_without_iva,sale_with_iva,iva_amount,total_cost,credit_note_value,client_name,city,salesperson_name,brand_name,category_name
0,99889,55054,2025-09-10,700M-014A,LIMA CADENA MOTOSIERRA 3/16 3/8,2,3.304348,3.800,0.495652,1.940652,NaN,CARVAJAL SEGUNDO,ISIDRO AYORA Y CHIRIBOGA ...,ADMINISTRADOR ...,VARIOS MOTORES,ACCESORIOS MOTOSIERRAS
1,99890,55055,2025-09-10,440-0001,ACEITE 2 TIEMPOS 120CC 589-57-20-02,1,1.565217,1.800,0.234783,1.238458,NaN,CONSUMIDOR,...,DE LA CUEVA HERNANDEZ KATERINE IRENE ...,HUSQVARNA,ACCESORIOS GUADAÑAS
2,99891,55056,2025-09-10,700M-622,BANDA MOTOCULTOR DUCATTI DRTL2100,2,22.608696,26.000,3.391304,15.750000,NaN,VEINTIMILLA RAMOS,OLMEDO LA CHIMBA ...,DE LA CUEVA HERNANDEZ KATERINE IRENE ...,AGROTA,REPUESTOS MOTOCULTORES
3,99892,55057,2025-09-10,700M-013A,BUJIA 4 TIEMPOS MECHA CORTA B7HS,1,3.090435,3.554,0.463565,1.720000,NaN,GAIBOR NUÑEZ,CAYAMBE ...,TALLER ...,VARIOS MOTORES,ACCESORIOS MOTORES 4T GASOLINA
4,99893,55057,2025-09-10,700M-077,MANGUERA DE COMBUSTIBLE 4T 1/4,10,7.500000,8.625,1.125000,200.000000,NaN,GAIBOR NUÑEZ,CAYAMBE ...,TALLER ...,TEMCO,"REPUESTOS MOTORES 4T GASOLINA, DIESEL"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440,100368,55190,2025-09-16,700B-162SI,LANZA 4 SALIDAS ACERO INOXIDABLE,4,212.000000,212.000,0.000000,184.000000,NaN,DENMAR S.A.,PANAMERICANA NORTE KM 47 ...,DE LA CUEVA HERNANDEZ JHONATAN MAURICIO ...,VARIOS BOMBAS,ACCESORIOS FUMIGACION
441,100369,55190,2025-09-16,700M-066,ACEITE 2TIEMPOS 1LITRO MARUYAMA,8,47.520000,54.648,7.128000,37.314421,NaN,DENMAR S.A.,PANAMERICANA NORTE KM 47 ...,DE LA CUEVA HERNANDEZ JHONATAN MAURICIO ...,MARUYAMA,ACCESORIOS MOTOSIERRAS
442,100370,55190,2025-09-16,700M-066,ACEITE 2TIEMPOS 1LITRO MARUYAMA,4,23.760000,27.324,3.564000,18.657211,NaN,DENMAR S.A.,PANAMERICANA NORTE KM 47 ...,DE LA CUEVA HERNANDEZ JHONATAN MAURICIO ...,MARUYAMA,ACCESORIOS MOTOSIERRAS
443,100371,55191,2025-09-16,1600-000,ACERO TRANSMISION 3 1/2 PULG (POLEA 3 1/2 ACERO),1,39.130435,45.000,5.869565,5.632000,NaN,ALBAN NUNEZ,QUITO ...,TALLER ...,TORNO,TORNO


## Products

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import pandas as pd
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# Load environment variables
load_dotenv(find_dotenv())

# SQL and MongoDB connections
engine = create_engine(os.environ['SQLSERVER_URI'])
client = MongoClient(os.environ['MONGO_URI'], server_api=ServerApi('1'))
cm_env = os.environ.get('CM_ENV', 'dev')

if cm_env == 'prod':
    collection = client["campomaq"]["cm_products"]
else:
    collection = client["campomaq_test"]["cm_products"]


# Read all products from SQL Server
df_products = pd.read_sql("""SELECT * FROM CM_PRODUCTOS cp 
WHERE cp.product_type = 'PO'
AND cp.category_name != 'SERVICIOS                                         '
AND cp.product_name not like '%VARIOS%'""", engine)

# Ensure product_id is integer (adjust column name as needed)
df_products['product_id'] = df_products['product_id'].astype(int)
df_products['category_id'] = df_products['category_id'].astype(int)
df_products['brand_id'] = df_products['brand_id'].astype(int)
df_products['brand_name'] = df_products['brand_name'].str.strip()
df_products['category_name'] = df_products['category_name'].str.strip()
df_products['product_code'] = df_products['product_code'].str.strip()
df_products['product_name'] = df_products['product_name'].str.strip()
df_products['product_type'] = df_products['product_type'].str.strip()
df_products['shelf_location'] = df_products['shelf_location'].str.strip()
df_products['is_spare_part'] = False


# # Upsert products into MongoDB efficiently
# for record in df_products.to_dict(orient="records"):
#     # Use product_id as unique key, update if exists, insert if not
#     collection.update_one(
#         {"product_id": record["product_id"]},
#         {"$set": record},
#         upsert=True
#     )

# print(f"✅ Upserted {len(df_products)} products into MongoDB.")

if not df_products.empty:
    # Convert DataFrame to JSON
    records = df_products.to_dict(orient="records")
        
    collection.insert_many(records)
    print(f"✅ Inserted {len(records)} records into MongoDB")

else:
    print("No new data to insert.")

✅ Inserted 4697 records into MongoDB


In [7]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4697 entries, 0 to 4696
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      4697 non-null   int64  
 1   product_code    4697 non-null   object 
 2   category_id     4697 non-null   int64  
 3   category_name   4697 non-null   object 
 4   product_name    4697 non-null   object 
 5   brand_id        4697 non-null   int64  
 6   brand_name      4697 non-null   object 
 7   shelf_location  4697 non-null   object 
 8   last_cost       4697 non-null   float64
 9   avg_cost        4697 non-null   float64
 10  price_cash      4697 non-null   float64
 11  price_credit    4697 non-null   float64
 12  price_card      4697 non-null   float64
 13  product_type    4697 non-null   object 
 14  iva             4697 non-null   bool   
 15  is_spare_part   4697 non-null   bool   
dtypes: bool(2), float64(5), int64(3), object(6)
memory usage: 523.0+ KB


## Stock

In [8]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import pandas as pd
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# Load environment variables
load_dotenv(find_dotenv())

# SQL and MongoDB connections
engine = create_engine(os.environ['SQLSERVER_URI'])
client = MongoClient(os.environ['MONGO_URI'], server_api=ServerApi('1'))
cm_env = os.environ.get('CM_ENV', 'dev')

if cm_env == 'prod':
    collection = client["campomaq"]["cm_stock"]
else:
    collection = client["campomaq_test"]["cm_stock"]


# Read all products from SQL Server
df_stock = pd.read_sql("SELECT * FROM CM_STOCK", engine)

# Ensure product_id is integer (adjust column name as needed)
df_stock['stock'] = df_stock['stock'].astype(int)

# Upsert products into MongoDB efficiently
for record in df_stock.to_dict(orient="records"):
    # Use product_id as unique key, update if exists, insert if not
    collection.update_one(
        {"product_code": record["product_code"]},
        {"$set": record},
        upsert=True
    )

print(f"✅ Upserted {len(df_stock)} products into MongoDB.")


✅ Upserted 4808 products into MongoDB.


In [9]:
# df_stock

## Products images

In [1]:
import os
from azure.storage.blob import ContainerClient

import pandas as pd
import io
from dotenv import load_dotenv, find_dotenv
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from collections import defaultdict

# Load environment variables
load_dotenv(find_dotenv())
AZ_SAS_URL =os.environ['AZ_CONTAINER_SAS_URL']

# 2. Connect to container
container_client = ContainerClient.from_container_url(AZ_SAS_URL)

# SQL and MongoDB connections
client = MongoClient(os.environ['MONGO_URI'], server_api=ServerApi('1'))
cm_env = os.environ.get('CM_ENV', 'dev')

cm_env = os.environ.get('CM_ENV', 'dev')

# Connect to MongoDB
if cm_env == 'prod':
    collection = client["campomaq"]["cm_images"]
else:
    collection = client["campomaq_test"]["cm_images"]

# 2. Connect to container
container_client = ContainerClient.from_container_url(AZ_SAS_URL)

# 3. List all blobs in the container
all_blobs = [blob.name for blob in container_client.list_blobs()]

# 4. Build image URLs with SAS token
image_urls = [f"{AZ_SAS_URL.split('?')[0]}/{blob}" for blob in all_blobs]
product_ids = [x.split('.')[0] for x in all_blobs]
product_ids = [x.split('_')[0] for x in product_ids]


# Create initial DataFrame
df = pd.DataFrame({
    'product_id': product_ids,
    'link': image_urls
})

# Filter for numeric product IDs only
df = df[df['product_id'].str.isnumeric()]

# Group by product_id and aggregate links into a list
df_image_names = df.groupby('product_id')['link'].apply(list).reset_index()
df_image_names['product_id'] = df_image_names['product_id'].astype(int)
df_image_names['update_time'] = pd.Timestamp.now()

# Sort by product_id
df_image_names = df_image_names.sort_values('product_id').reset_index(drop=True)

# Upsert products into MongoDB efficiently
# for record in df_image_names.to_dict(orient="records"):
#     # Use product_id as unique key, update if exists, insert if not
#     collection.update_one(
#         {"product_id": record["product_id"]},
#         {"$set": record},
#         upsert=True
#     )

# print(f"✅ Upserted {len(df_image_names)} products into MongoDB.")


In [11]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
# df_image_names.head(10)

In [3]:
if not df_image_names.empty:
    # Convert DataFrame to JSON
    records = df_image_names.to_dict(orient="records")
        
    collection.insert_many(records)
    print(f"✅ Inserted {len(records)} records into MongoDB")

else:
    print("No new data to insert.")

✅ Inserted 2387 records into MongoDB
